# CBED Frame Simulation

Demonstrate how to simulate a CBED frame using this package, with pseudo-realistic and controllable quality.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from cbed_simulation import OrientedPhase, ExperimentInformation, FrameParameters
from cbed_simulation.crystal_orientation import LatticeMultipliers

We load a structure in a given orientation, this can be an `hkl` index or an `orix.quaternion.Rotation` object for an aribtrary orientation. The convention used for rotations is `crystal2lab`.

In [ ]:
phase = OrientedPhase.from_cif(
    "./tests/Si.cif",
    zone_axis=(1, 1, 1),
)

The `ExperimentInformation` sets the physical parameters of the simulation, notably the detector shape and the pixel scale factor.

`complex` numbers are used for coordinates, the real part is the horizontal right-positive axis, while the imaginary part is the vertical down-positive axis (i.e. `matplotlib` image display convention).

In [ ]:
experiment = ExperimentInformation.from_tem_params(
    camera_length_m=0.6,
    semiconv_mrad=2.3,
    frame_shape=(512, 512),
    voltage_kv=200,
    pixelsize_um=125.,
    centre_px=complex(225, 275)
)

In [ ]:
experiment.radius_px

In [ ]:
experiment.pattern_scale_factor

We simulate the peak positions using `phase.peak_positions`, by default this uses kinematical simulation. It is possible to strain the crystal lattice when generating `peak_positions`.

In [ ]:
sim_peaks = phase.peak_positions(experiment)

A `FrameParameters` object defines how the spot positions are composited into a numpy array image, via the `phase.synthetic` method:

In [ ]:
frame_params = FrameParameters(current_pa=12., texture_strength=0.9)
sim_frame = phase.synthetic(experiment, sim_peaks, frame_params=frame_params)

The default `FrameParameters` are intended to create a "reasonable" default CBED frame. The result is not physically real, but rather aesthetically correct for a "modern" TEM camera and geometrically correct with respect to the input peak positions.

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 6))
ax.imshow(sim_frame, cmap="gray")
ax.set_title("CBED Frame");

The `FrameParameters` object contains parameters to adjust properties of the image like intensity and noise level:

In [ ]:
frame_params = FrameParameters(
    current_pa=1.,
    intensity_raw_power=2.,
    psf_sigma=0.,
    additive_noise_scale=0.25,
)
sim_frame = phase.synthetic(experiment, sim_peaks, frame_params=frame_params)

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 6))
ax.imshow(sim_frame, cmap="gray")
ax.set_title("1 pA CBED Frame");

We can also do the opposite and remove all noise:

In [ ]:
frame_params = FrameParameters(
    textured=False,
    poisson_frame=False,
    psf_sigma=0.,
    inelastic_scatter_sigma=0.,
    additive_noise_scale=0.,
)
sim_frame = phase.synthetic(experiment, sim_peaks, frame_params=frame_params)

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 6))
ax.imshow(sim_frame, cmap="gray")
ax.set_title("Clean CBED Frame");

We can strain the lattice during the call to `peak_positions`:

In [ ]:
mult = LatticeMultipliers(a=1.02, b=1.02, c=1.02)
sim_peaks_strained = phase.peak_positions(experiment, lattice_mod=mult)

In [ ]:
frame_params = FrameParameters(textured=False)
sim_frame = phase.synthetic(experiment, sim_peaks, frame_params=frame_params)
sim_frame_strained = phase.synthetic(experiment, sim_peaks_strained, frame_params=frame_params)

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 6))
ax.imshow(sim_frame - sim_frame_strained, cmap="bwr")
ax.set_title("Uniform Strained - Unstrained difference");